In [20]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [21]:
df = pd.read_excel("online_course_recommendation_v2.xlsx")

In [22]:
df.head()

df.info()

df.isnull().sum()

df.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   user_id                   100000 non-null  int64  
 1   course_id                 100000 non-null  int64  
 2   course_name               100000 non-null  object 
 3   instructor                100000 non-null  object 
 4   course_duration_hours     100000 non-null  float64
 5   certification_offered     100000 non-null  object 
 6   difficulty_level          100000 non-null  object 
 7   rating                    100000 non-null  float64
 8   enrollment_numbers        100000 non-null  int64  
 9   course_price              100000 non-null  float64
 10  feedback_score            100000 non-null  float64
 11  study_material_available  100000 non-null  object 
 12  time_spent_hours          100000 non-null  float64
 13  previous_courses_taken    100000 non-null  in

np.int64(0)

In [23]:
df = df.drop_duplicates(subset='course_name').reset_index(drop=True)

In [24]:
print(df.shape)
print(df['course_name'].nunique())

(20, 14)
20


In [28]:
df['features'] = (
    df['course_name'].astype(str) + " " +
    df['instructor'].astype(str) + " " +
    df['difficulty_level'].astype(str) + " " +
    df['certification_offered'].astype(str) + " " +
    df['study_material_available'].astype(str)
)

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['features'])

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(tfidf_matrix)

In [34]:
def recommend(course_name):
    idx = df[df['course_name'] == course_name].index[0]

    sim_scores = list(enumerate(similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Remove the selected course itself
    sim_scores = sim_scores[1:6]

    course_indices = [i[0] for i in sim_scores]

    return df.iloc[course_indices][[
        'course_name',
        'instructor',
        'difficulty_level',
        'rating'
    ]]

In [47]:
recommend("Python for Beginners")

,course_name,instructor,difficulty_level,rating
17,Stock Market and Trading Strategies,Emma Harris,Beginner,4.9
1,Cybersecurity for Professionals,Alexander Young,Beginner,4.3
5,Networking and System Administration,Dr. Robert Davis,Beginner,4.9
4,Ethical Hacking Masterclass,Daniel White,Beginner,2.8
8,Graphic Design with Canva,Alexander Young,Beginner,4.5


In [48]:
print(df[['course_name', 'instructor', 'difficulty_level',
          'certification_offered', 'study_material_available']])

                                  course_name        instructor  \
0                        Python for Beginners       Emma Harris   
1             Cybersecurity for Professionals   Alexander Young   
2            DevOps and Continuous Deployment    Dr. Mia Walker   
3             Project Management Fundamentals    Benjamin Lewis   
4                 Ethical Hacking Masterclass      Daniel White   
5        Networking and System Administration  Dr. Robert Davis   
6        Personal Finance and Wealth Building    Benjamin Lewis   
7   Blockchain and Decentralized Applications    Benjamin Lewis   
8                   Graphic Design with Canva   Alexander Young   
9              Fitness and Nutrition Coaching        Liam Adams   
10                    Public Speaking Mastery    Benjamin Lewis   
11              Photography and Video Editing      Daniel White   
12                  Advanced Machine Learning      Daniel White   
13                Game Development with Unity  Dr. Robert Davi

In [49]:
print(df['course_name'].tolist())

['Python for Beginners', 'Cybersecurity for Professionals', 'DevOps and Continuous Deployment', 'Project Management Fundamentals', 'Ethical Hacking Masterclass', 'Networking and System Administration', 'Personal Finance and Wealth Building', 'Blockchain and Decentralized Applications', 'Graphic Design with Canva', 'Fitness and Nutrition Coaching', 'Public Speaking Mastery', 'Photography and Video Editing', 'Advanced Machine Learning', 'Game Development with Unity', 'Cloud Computing Essentials', 'Mobile App Development with Swift', 'Data Visualization with Tableau', 'Stock Market and Trading Strategies', 'Fundamentals of Digital Marketing', 'AI for Business Leaders']


knn

In [50]:
from sklearn.neighbors import NearestNeighbors

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['features'])

In [52]:
knn_model = NearestNeighbors(
    n_neighbors=6,
    metric='cosine',
    algorithm='brute'
)

knn_model.fit(tfidf_matrix)

NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=6)

In [53]:
def recommend_knn(course_name):

    idx = df[df['course_name'] == course_name].index[0]

    distances, indices = knn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=6
    )

    recommendations = df.iloc[indices[0][1:]][[
        'course_name',
        'instructor',
        'difficulty_level',
        'rating'
    ]]

    return recommendations

In [54]:
recommend_knn("Python for Beginners")

,course_name,instructor,difficulty_level,rating
17,Stock Market and Trading Strategies,Emma Harris,Beginner,4.9
1,Cybersecurity for Professionals,Alexander Young,Beginner,4.3
5,Networking and System Administration,Dr. Robert Davis,Beginner,4.9
8,Graphic Design with Canva,Alexander Young,Beginner,4.5
4,Ethical Hacking Masterclass,Daniel White,Beginner,2.8
